In [86]:
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.chains.question_answering import load_qa_chain
import warnings
import json
import pandas as pd
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import os
import openai
import re
from jobspy import scrape_jobs

In [50]:
pdf = 'content/resume.pdf'
pdf_reader = PdfReader(pdf)
print(pdf_reader)

In [51]:
text = ''
for page in pdf_reader.pages:
    text += page.extract_text()

print(text)

Andre Sealy
New York, NY |(347)-461-7821 |andretsealy@gmail.com |/ewww.kidquant.com |/gtbkidquant
EDUCATION
Stevens Institute of Technology New York, NY
Masters of Science in Financial Engineering; Major GPA: 3.81 Sept 2024 - Present
Relevant Coursework: Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, Deep Learning
Hunter College New York, NY
Bachelor of Arts in Mathematics; Major GPA: 3.68 Jan 2020 - Present
Relevant Coursework: Numerical Analysis, Real Analysis, Mathematical Statistics, Linear Algebra
Pace University New York, NY
Bachelor of Business Administration in Finance; Major GPA: 4.0 December 2018
Bachelor of Arts in Economics; Major GPA: 4.0
Honors: Beta Gamma Sigma, Golden Key Society
EXPERIENCE
America On Tech New York, NY
Data Science Instructor Nov 2023 - Present
•Facilitate and coordinate weekly lectures, lab projects, coding examples, graded quizzes and homework assignments.
• Design interactive weekly learning modules for more than 50 stu

In [52]:
# split the long text into small chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700,
                                               chunk_overlap=200,
                                               length_function=len)

chunks = text_splitter.split_text(text)
chunks

['Andre Sealy\nNew York, NY |(347)-461-7821 |andretsealy@gmail.com |/ewww.kidquant.com |/gtbkidquant\nEDUCATION\nStevens Institute of Technology New York, NY\nMasters of Science in Financial Engineering; Major GPA: 3.81 Sept 2024 - Present\nRelevant Coursework: Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, Deep Learning\nHunter College New York, NY\nBachelor of Arts in Mathematics; Major GPA: 3.68 Jan 2020 - Present\nRelevant Coursework: Numerical Analysis, Real Analysis, Mathematical Statistics, Linear Algebra\nPace University New York, NY\nBachelor of Business Administration in Finance; Major GPA: 4.0 December 2018\nBachelor of Arts in Economics; Major GPA: 4.0',
 'Pace University New York, NY\nBachelor of Business Administration in Finance; Major GPA: 4.0 December 2018\nBachelor of Arts in Economics; Major GPA: 4.0\nHonors: Beta Gamma Sigma, Golden Key Society\nEXPERIENCE\nAmerica On Tech New York, NY\nData Science Instructor Nov 2023 - Present\n•Faci

In [53]:
chunks[0]

'Andre Sealy\nNew York, NY |(347)-461-7821 |andretsealy@gmail.com |/ewww.kidquant.com |/gtbkidquant\nEDUCATION\nStevens Institute of Technology New York, NY\nMasters of Science in Financial Engineering; Major GPA: 3.81 Sept 2024 - Present\nRelevant Coursework: Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, Deep Learning\nHunter College New York, NY\nBachelor of Arts in Mathematics; Major GPA: 3.68 Jan 2020 - Present\nRelevant Coursework: Numerical Analysis, Real Analysis, Mathematical Statistics, Linear Algebra\nPace University New York, NY\nBachelor of Business Administration in Finance; Major GPA: 4.0 December 2018\nBachelor of Arts in Economics; Major GPA: 4.0'

In [54]:

load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]

def openai_function(openai_api_key, chunks, analyze):

    # Using OpenAI service for embedding
    embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

    # Facebook AI Similarity Search library help us to convert text data to numerical vector
    vectorstores = FAISS.from_texts(chunks, embedding=embeddings)

    # compares the query and chunks, enabling the selection of the top 'K' most similiar chunks based on their similarity scores.
    docs = vectorstores.similarity_search(query=analyze, k=3)

    # creates an OpenAI object, using the ChatGPT 4 
    llm = ChatOpenAI(model='gpt-4o', api_key=openai_api_key)

    # question-answering (QA) pipeline, making use of the load_qa_chain function
    chain = load_qa_chain(llm=llm, chain_type='stuff')

    response = chain.run(input_documents=docs, question=analyze)
    return response



In [55]:
def resume_summary(query_with_chunks):
    query = f''' need to detailed summarization of below resume and finally conclude them

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

summary = resume_summary(query_with_chunks=chunks)
summary_result = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=summary)
print(summary_result)

Andre Sealy is a highly educated and experienced professional based in New York, NY. He is currently pursuing a Master's degree in Financial Engineering at Stevens Institute of Technology, maintaining a major GPA of 3.81, with coursework covering stochastic calculus, pricing & hedging, probability theory, machine learning, and deep learning. He also holds a Bachelor of Arts in Mathematics from Hunter College with a major GPA of 3.68, and a Bachelor of Business Administration in Finance, as well as a Bachelor of Arts in Economics, both from Pace University with a perfect major GPA of 4.0. During his time at Pace, he was honored with memberships in Beta Gamma Sigma and the Golden Key Society.

Professionally, Andre is working as a Data Science Instructor at America On Tech, where he designs and facilitates educational content focused on machine learning, statistics, and data visualization for over 50 students. Previously, he worked in fixed income data research at Alliance Bernstein, whe

In [ ]:
def resume_strength(query_with_chunks):
    query = f'''need to detailed analysis and explain of the strength of below resume and finally conclude them
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

strengths = resume_strength(query_with_chunks=chunks)
strengths_result = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=strengths)
print(strengths_result)

In [ ]:
def resume_weakness(query_with_chunks):
    query = f'''need to detailed analysis and explain of the weakness of below resume and how to improve make a better resume.

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

weakness = resume_weakness(query_with_chunks=summary_result)
result_weakness = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=weakness)
print(result_weakness)

In [ ]:
def job_title_suggestion(query_with_chunks):

    query = f''' what are the job roles i apply to likedin based on below?
                  
                  """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                  {query_with_chunks}
                  """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

suggestion = job_title_suggestion(query_with_chunks=summary_result)
result_suggestion = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=suggestion)
print(result_suggestion)

In [56]:
def job_title_prompt(query_with_chunks):
    query = f'''Based on my resume, come up with some job roles that would best fit my skills and abilities.
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

job_prompt = job_title_prompt(query_with_chunks=summary_result)
jobs = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=job_prompt)
print(jobs)


Based on your resume, here are some job roles that would best fit your skills and abilities:

1. **Quantitative Analyst**: Your background in financial engineering, mathematics, and experience with quantitative investment strategies make you a strong candidate for roles that require analyzing and developing quantitative models for trading and risk management.

2. **Data Scientist in Finance**: Your proficiency in machine learning, statistics, and programming, combined with your experience in predicting user behavior and analyzing financial data, would be valuable in data-driven decision-making roles in the finance sector.

3. **Financial Engineer**: Your coursework in stochastic calculus, pricing & hedging, and deep learning, along with your practical experience, align well with roles focused on developing financial models and algorithms for pricing and risk assessment.

4. **Investment Research Analyst**: With your experience in fixed income data research and producing research report

In [89]:
jobs_list = re.findall(r"\*\*(.*?)\*\*", jobs)

In [91]:
# Loop over the strings in jobs_list and run the function scrape_jobs
for job in jobs_list:
    # Scrape jobs for each job title in jobs_list
    jobs_scraped_for_job = scrape_jobs(
        site_name=[
            "indeed",
            "linkedin",
            "glassdoor",
            "google",
        ],
        search_term=job,
        location="New York, NY",
        max_results=5,
        country_indeed="USA",
    )

    jobs_scraped = pd.DataFrame()

    jobs_scraped_for_job['job_type'] = job

    # Merge the output with the previous dataframe
    jobs_scraped = pd.concat([jobs_scraped, jobs_scraped_for_job], ignore_index=True)


jobs_scraped = jobs_scraped[
    [
        # "id",
        "site",
        "job_url",
        "title",
        "company",
        "date_posted",
        "job_type",
        "interval",
        "min_amount",
        "max_amount",
        "currency",
        "description",
    ]
]

jobs_scraped.head()

,site,job_url,title,company,date_posted,job_type,interval,min_amount,max_amount,currency,description
0,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,"SVP, HR ACT - Quality Assurance, Monitoring & ...",Citi,2025-04-29,Risk Management Analyst,yearly,163600.0,245400.0,USD,"Individuals in Quality Assurance, Monitoring \..."
1,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,"VP, HR ACT - Quality Assurance, Monitoring & T...",Citi,2025-04-29,Risk Management Analyst,yearly,129840.0,194760.0,USD,"Individuals in Quality Assurance, Monitoring \..."
2,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Fraud Strategy Analyst,Prove,2025-04-29,Risk Management Analyst,yearly,100000.0,115000.0,USD,"New York, NY\n**About Prove**\n===============..."
3,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Regulatory and Compliance Analyst,pillbag pharmacy inc,2025-04-28,Risk Management Analyst,hourly,20.0,35.0,USD,**Overview** \nPillbag Pharmacy inc. is a com...
4,glassdoor,https://www.glassdoor.com/job-listing/j?jl=100...,Senior Analyst - Enterprise Control Management...,Amex,2025-04-28,Risk Management Analyst,yearly,55000.0,105000.0,USD,**You Lead the Way. We’ve Got Your Back.**\n\n...


In [94]:
skills = jobs_scraped[['title', 'description', 'job_type']]

In [95]:
# Convert the DataFrame to JSON
jobs_json = jobs_scraped.to_json(orient='records')

# Print the JSON string to verify
print(jobs_json)

[{"site":"glassdoor","job_url":"https:\/\/www.glassdoor.com\/job-listing\/j?jl=1009725108310","title":"SVP, HR ACT - Quality Assurance, Monitoring & Testing Sr. Lead Analyst","company":"Citi","date_posted":1745884800000,"job_type":"Risk Management Analyst","interval":"yearly","min_amount":163600.0,"max_amount":245400.0,"currency":"USD","description":"Individuals in Quality Assurance, Monitoring \\& Testing are responsible for the assessment of outcomes from activities and processes against conformance with applicable requirements to strengthen risk management quality such as quality testing performed for business function quality control and transformation lead quality control post completion of an activity\/process. This includes the development and execution of Monitoring and Testing for controls, such as control design assessment, design of operational effectiveness for monitoring \\& testing tools, monitoring\/testing design assessment, and execution of monitoring\/testing tools to

In [97]:
def openai_function(openai_api_key, chunks, analyze):

    # Using OpenAI service for embedding
    embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

    # Facebook AI Similarity Search library help us to convert text data to numerical vector
    vectorstores = FAISS.from_texts(chunks, embedding=embeddings)

    # compares the query and chunks, enabling the selection of the top 'K' most similiar chunks based on their similarity scores.
    docs = vectorstores.similarity_search(query=analyze, k=3)

    # creates an OpenAI object, using the ChatGPT 4 
    llm = ChatOpenAI(model='gpt-4.1-mini', api_key=openai_api_key)

    # question-answering (QA) pipeline, making use of the load_qa_chain function
    chain = load_qa_chain(llm=llm, chain_type='stuff')

    response = chain.run(input_documents=docs, question=analyze)
    return response



In [98]:
def analyze_skills_from_jobs(jobs_json):
    query = f'''Analyze the following job data to extract the most important skills for candidates:
                \"\"\"{jobs_json}\"\"\"
             '''
    return query

# Example usage
skills_query = analyze_skills_from_jobs(jobs_json)
skills_result = openai_function(openai_api_key=openai.api_key, chunks=[jobs_json], analyze=skills_query)

# Print the result to see the extracted skills
print(skills_result)

Based on the analysis of the provided job data for Risk Management Analyst roles from various companies and sources, the most important skills and qualifications for candidates include:

1. **Risk Management and Control Expertise**
   - Strong knowledge of operational risk management, compliance, audit, and control-related functions.
   - Ability to identify, measure, manage, and remediate key risks and controls.
   - Experience with risk control frameworks, policies, standards, and procedures.
   - Familiarity with Governance, Risk, and Compliance (GRC) tools (e.g., RSA Archer, ServiceNow).
   - Experience in conducting risk assessments, control testing, and issue management.
   - Understanding of regulatory requirements and compliance laws (e.g., DEA, FDA, HIPAA, CMS, SOX, FFIEC, PCI-DSS, GDPR).
   - Ability to support and manage regulatory exams, audits, and internal controls.
   - Knowledge of model risk management and model validation.

2. **Analytical and Quantitative Skills**
  